## A notebook to create a bar graph of CTs inside AS

## Install and import libraries

In [456]:

%pip install pandas

import pandas as pd
import requests
from  io import StringIO
from pprint import pprint

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Global settings

In [457]:
hra_pop_version = 'v0.12.0'
branch = 'v0.12.0'

output_folder = 'output/ctann-tree'

## Load hra-pop data as `df`

In [458]:
# could also use https://apps.humanatlas.io/api/grlc/hra-pop.html#get-/cell-types-in-atlas
url = f"https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/atlas-ad-hoc/cell-types-in-anatomical-structurescts-per-as.csv"

headers = {
    'Accept': 'text/csv'
}

data = requests.get(url=url, headers=headers).text

df = pd.read_csv(StringIO(data))
df

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count
0,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002079,ductal,15.312,0.522523,1
1,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002064,acinar,8.640,0.294840,1
2,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000115,endothelial,3.864,0.131859,1
3,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000738,immune,1.464,0.049959,1
4,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002410,activated_stellate,0.024,0.000819,1
...,...,...,...,...,...,...,...,...,...,...,...
9216,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000097,Mast Cell,15322.464,0.024702,1
9217,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033039,CD8+ T Cell,3691.176,0.005951,1
9218,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_lymphatic-endo...,Lymphatic Endothelial (and some immune cells),1753.956,0.002828,1
9219,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_basal-epitheli...,Basal Epithelial Cell,970.104,0.001564,1


## Get updated CT level mapping

In [459]:
with open('data/ad_hoc_cell_type_level_mapping_query.rq', 'r') as f:
    query = f.read()
    
# define endpoint
url = "https://lod.humanatlas.io/sparql"

# define parameters
params = {
    "query": query,
}

# set header
headers = {
    "Accept": "text/csv"
}

# Send the GET request
response = requests.get(url, headers=headers, params=params)

# convert text to file-like object
csv_data = StringIO(response.text)

# concert to DataFrame
yasgui_cell_types_level_mapping = pd.read_csv(csv_data)
yasgui_cell_types_level_mapping

,cell_label,cell_id,level_1_cell_id,level_1_cell_label
0,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
1,fibroblast,http://purl.obolibrary.org/obo/CL_0000057,http://purl.obolibrary.org/obo/CL_0000499,stromal cell
2,epithelial cell,http://purl.obolibrary.org/obo/CL_0000066,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
3,blood vessel endothelial cell,http://purl.obolibrary.org/obo/CL_0000071,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
4,mesothelial cell,http://purl.obolibrary.org/obo/CL_0000077,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
...,...,...,...,...
190,lung interstitial macrophage,http://purl.obolibrary.org/obo/CL_4033043,http://purl.obolibrary.org/obo/CL_0000235,macrophage
191,deuterosomal cell,http://purl.obolibrary.org/obo/CL_4033044,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
192,lung migratory dendritic cell,http://purl.obolibrary.org/obo/CL_4033045,http://purl.obolibrary.org/obo/CL_0000451,dendritic cell
193,respiratory suprabasal cell,http://purl.obolibrary.org/obo/CL_4033048,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell


In [460]:
# yasgui_cell_types_level_mapping['level_1_cell_id'] = yasgui_cell_types_level_mapping['level_1_cell_id'].apply(
#     lambda id: id.split('/')[-1].replace('_', ':'))
yasgui_cell_types_level_mapping

,cell_label,cell_id,level_1_cell_id,level_1_cell_label
0,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
1,fibroblast,http://purl.obolibrary.org/obo/CL_0000057,http://purl.obolibrary.org/obo/CL_0000499,stromal cell
2,epithelial cell,http://purl.obolibrary.org/obo/CL_0000066,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
3,blood vessel endothelial cell,http://purl.obolibrary.org/obo/CL_0000071,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
4,mesothelial cell,http://purl.obolibrary.org/obo/CL_0000077,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
...,...,...,...,...
190,lung interstitial macrophage,http://purl.obolibrary.org/obo/CL_4033043,http://purl.obolibrary.org/obo/CL_0000235,macrophage
191,deuterosomal cell,http://purl.obolibrary.org/obo/CL_4033044,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
192,lung migratory dendritic cell,http://purl.obolibrary.org/obo/CL_4033045,http://purl.obolibrary.org/obo/CL_0000451,dendritic cell
193,respiratory suprabasal cell,http://purl.obolibrary.org/obo/CL_4033048,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell


## Join `df` containing `hra-pop` CTs with new cell-mapping

In [461]:
df_temp = df

# remove ASCTB TEMP
df_temp = df_temp[~df_temp['cell_id'].str.contains('ASCT', na=False)]
df_temp



,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count
0,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002079,ductal,15.312,0.522523,1
1,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002064,acinar,8.640,0.294840,1
2,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000115,endothelial,3.864,0.131859,1
3,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000738,immune,1.464,0.049959,1
4,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002410,activated_stellate,0.024,0.000819,1
...,...,...,...,...,...,...,...,...,...,...,...
9213,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000115,Endothelial,40223.460,0.064846,1
9215,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033038,CD4+ T Cell,16776.624,0.027046,1
9216,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000097,Mast Cell,15322.464,0.024702,1
9217,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033039,CD8+ T Cell,3691.176,0.005951,1


In [462]:
# adjust cell_id column
# df_temp['cell_id'] = df_temp['cell_id'].apply(lambda id: id.split('/')[-1].replace('_', ':'))
df_temp

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count
0,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002079,ductal,15.312,0.522523,1
1,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002064,acinar,8.640,0.294840,1
2,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000115,endothelial,3.864,0.131859,1
3,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000738,immune,1.464,0.049959,1
4,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002410,activated_stellate,0.024,0.000819,1
...,...,...,...,...,...,...,...,...,...,...,...
9213,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000115,Endothelial,40223.460,0.064846,1
9215,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033038,CD4+ T Cell,16776.624,0.027046,1
9216,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000097,Mast Cell,15322.464,0.024702,1
9217,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033039,CD8+ T Cell,3691.176,0.005951,1


In [463]:
# Merge look-up df with df
df_temp = df.merge(
    yasgui_cell_types_level_mapping[['cell_id', 'level_1_cell_id', 'level_1_cell_label']],
    left_on='cell_id',  # Column in main df
    right_on='cell_id',  # Column in lookup df
    how='left'      # Keep all rows from main df
)

df_temp

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,level_1_cell_id,level_1_cell_label
0,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002079,ductal,15.312,0.522523,1,NaN,NaN
1,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002064,acinar,8.640,0.294840,1,NaN,NaN
2,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000115,endothelial,3.864,0.131859,1,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
3,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000738,immune,1.464,0.049959,1,NaN,NaN
4,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002410,activated_stellate,0.024,0.000819,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9370,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000097,Mast Cell,15322.464,0.024702,1,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
9371,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033039,CD8+ T Cell,3691.176,0.005951,1,NaN,NaN
9372,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_lymphatic-endo...,Lymphatic Endothelial (and some immune cells),1753.956,0.002828,1,NaN,NaN
9373,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_basal-epitheli...,Basal Epithelial Cell,970.104,0.001564,1,NaN,NaN


In [464]:
# handle missing values
df_temp['level_1_cell_id'] = df_temp['level_1_cell_id'].fillna(
    'No higher-level CT')
df_temp['level_1_cell_label'] = df_temp['level_1_cell_label'].fillna(
    'No higher-level CT')

In [465]:
df = df_temp
df

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,level_1_cell_id,level_1_cell_label
0,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002079,ductal,15.312,0.522523,1,No higher-level CT,No higher-level CT
1,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002064,acinar,8.640,0.294840,1,No higher-level CT,No higher-level CT
2,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000115,endothelial,3.864,0.131859,1,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
3,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000738,immune,1.464,0.049959,1,No higher-level CT,No higher-level CT
4,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002410,activated_stellate,0.024,0.000819,1,No higher-level CT,No higher-level CT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9370,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000097,Mast Cell,15322.464,0.024702,1,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
9371,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033039,CD8+ T Cell,3691.176,0.005951,1,No higher-level CT,No higher-level CT
9372,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_lymphatic-endo...,Lymphatic Endothelial (and some immune cells),1753.956,0.002828,1,No higher-level CT,No higher-level CT
9373,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_basal-epitheli...,Basal Epithelial Cell,970.104,0.001564,1,No higher-level CT,No higher-level CT


In [466]:
# keep unique combinations of tool, cell_id, cell_label, level_1_cell_id, and level_1_cell_label
df_unique = df.drop_duplicates(subset=['tool', 'cell_id', 'cell_label', 'level_1_cell_id', 'level_1_cell_label'])

# adjust format for ontology ID
for col in ['cell_id', 'level_1_cell_id']:
  df_unique[col] = df[col].apply(lambda x: x.split('/')[-1].replace('_', ':') if isinstance(x, str) else x)

df_unique

C:\Users\abueckle\AppData\Local\Temp\1\ipykernel_15864\4200318101.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique[col] = df[col].apply(lambda x: x.split('/')[-1].replace('_', ':') if isinstance(x, str) else x)


,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,level_1_cell_id,level_1_cell_label
0,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0002079,ductal,15.312,0.522523,1,No higher-level CT,No higher-level CT
1,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0002064,acinar,8.640,0.294840,1,No higher-level CT,No higher-level CT
2,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0000115,endothelial,3.864,0.131859,1,CL:0000115,endothelial cell
3,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0000738,immune,1.464,0.049959,1,No higher-level CT,No higher-level CT
4,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0002410,activated_stellate,0.024,0.000819,1,No higher-level CT,No higher-level CT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9363,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,CL:0002062,Type 1 Alveolar Epithelial Cell,292533.168,0.471605,1,CL:0000066,epithelial cell
9364,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,ASCTB-TEMP:mpo-,MPO+,93194.724,0.150243,1,No higher-level CT,No higher-level CT
9370,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,CL:0000097,Mast Cell,15322.464,0.024702,1,CL:0000988,hematopoietic cell
9372,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,ASCTB-TEMP:lymphatic-endothelial-and-some-immu...,Lymphatic Endothelial (and some immune cells),1753.956,0.002828,1,No higher-level CT,No higher-level CT


## Load and enrich SLIM for vis in ASCT+B Reporter, join with `df`

In [467]:
# sheet_url = 'https://docs.google.com/spreadsheets/d/1JZE9BprxatUUopN25P1G16flGXvbuRipuddRFREkN4A/edit?gid=0#gid=0'
sheet = pd.read_csv('data/SLIM hierarchy.csv', skiprows=2)
df_slim = sheet.fillna("")
df_slim

,AS/1,AS/1/LABEL,AS/1/ID,AS/2,AS/2/LABEL,AS/2/ID,AS/3,AS/3/LABEL,AS/3/ID
0,cell,cell,CL:0000000,connective tissue cell,connective tissue cell,CL:0002320,adipocyte,adipocyte,CL:0000136
1,cell,cell,CL:0000000,melanocyte,melanocyte,CL:0000148,,,
2,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,monocyte,monocyte,CL:0000576
3,cell,cell,CL:0000000,,,,exocrine cell,exocrine cell,CL:0000152
4,cell,cell,CL:0000000,extraembryonic cell,extraembryonic cell,CL:0000349,,,
5,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,,,
6,cell,cell,CL:0000000,germ line cell,germ line cell,CL:0000039,,,
7,cell,cell,CL:0000000,bone cell,bone cell,CL:0001035,,,
8,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,blood cell,blood cell,CL:0000081
9,cell,cell,CL:0000000,neural cell,neural cell,CL:0002319,glial cell,glial cell,CL:0000125


In [468]:
# Make a copy of df_slim to preserve the original
df_slim_result = df_slim.copy()

# Ensure key columns are treated as strings for safe matching
df_slim['AS/2/ID'] = df_slim['AS/2/ID'].astype(str)
df_slim['AS/3/ID'] = df_slim['AS/3/ID'].astype(str)
df_unique['level_1_cell_id'] = df_unique['level_1_cell_id'].astype(str)

for _, row in df_unique.iterrows():
    level_1_cell_id = str(row['level_1_cell_id'])
    # Replace with your actual column name
    level_1_cell_label = row['level_1_cell_label']

    # Then use `cell_id` as before to match in df_slim
    mask = (df_slim['AS/2/ID'] == level_1_cell_id) | (df_slim['AS/3/ID'] == level_1_cell_id)
    matching_rows = df_slim[mask].copy()

    if not matching_rows.empty:
        matching_rows['AS/4'] = str(row['cell_label'])
        matching_rows['AS/4/LABEL'] = str(row['cell_label'])
        matching_rows['AS/4/ID'] = str(row['cell_id'])
        df_slim_result = pd.concat([df_slim_result, matching_rows], ignore_index=True)

# Remove original rows in df_slim if there was at least one match
# Step 1: Identify matched cell IDs
matched_ids = set(df_slim_result['AS/4/ID'].dropna().astype(str))

# Step 2: Remove original rows (no AS/4/ID) that matched either AS/2/ID or AS/3/ID
df_slim_result = df_slim_result[
    ~(
        df_slim_result['AS/4/ID'].isna() & (
            df_slim_result['AS/2/ID'].isin(matched_ids) |
            df_slim_result['AS/3/ID'].isin(matched_ids)
        )
    )
]

df_slim_result

C:\Users\abueckle\AppData\Local\Temp\1\ipykernel_15864\3065953708.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique['level_1_cell_id'] = df_unique['level_1_cell_id'].astype(str)


,AS/1,AS/1/LABEL,AS/1/ID,AS/2,AS/2/LABEL,AS/2/ID,AS/3,AS/3/LABEL,AS/3/ID,AS/4,AS/4/LABEL,AS/4/ID
3,cell,cell,CL:0000000,,,,exocrine cell,exocrine cell,CL:0000152,NaN,NaN,NaN
4,cell,cell,CL:0000000,extraembryonic cell,extraembryonic cell,CL:0000349,,,,NaN,NaN,NaN
5,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,,,,NaN,NaN,NaN
6,cell,cell,CL:0000000,germ line cell,germ line cell,CL:0000039,,,,NaN,NaN,NaN
7,cell,cell,CL:0000000,bone cell,bone cell,CL:0001035,,,,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
545,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,blood cell,blood cell,CL:0000081,Mast Cell,Mast Cell,CL:0000097
546,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,macrophage,macrophage,CL:0000235,Mast Cell,Mast Cell,CL:0000097
547,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,dendritic cell,dendritic cell,CL:0000451,Mast Cell,Mast Cell,CL:0000097
548,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,B cell,B cell,CL:0000236,Mast Cell,Mast Cell,CL:0000097


## Export

In [469]:
df_slim_result.to_csv('output/ctann_tree.csv', index=False)
# put on Google Sheets: https://docs.google.com/spreadsheets/d/1ISKJOktR6pl6uUNhLdV6xcXMZAQ3KfEgf8BMp9Jjs9s/edit?gid=1266919613#gid=1266919613